# Pilot: which model can write correct options? (no training, about 30 to 45 min)

Fine-tuning mostly teaches **format, style and difficulty**. Whether the options and answer are *correct* depends on what the
model already knows from pretraining. FLAN-T5-base (250M) did not know enough, so no amount of fine-tuning fixed it.

So before spending hours training, this notebook tests three instruct models **as they are** (2 dataset examples in the prompt,
no training) on the same 30 test inputs:
- Qwen2.5-1.5B-Instruct, Qwen2.5-3B-Instruct, Qwen2.5-7B-Instruct (all loaded in 4-bit, so even 7B fits on a free T4)

Each MCQ goes through two modern checks:
1. **Strict format check:** 4 distinct options, one answer letter.
2. **Self-verification:** the model answers its own question without seeing the key. If its answer differs from the key, the MCQ is rejected
   (this catches most wrong answer keys).

At the end you get a table (valid %, self-check pass %, seconds per MCQ) and a CSV to grade by hand. **We fine-tune only the smallest
model that is already mostly correct here.**

Colab: Runtime → T4 GPU, `btech_mcq_dataset_clean.zip` in Drive folder `adaptive-quiz`. Kaggle: GPU T4, Internet ON, dataset added.

In [ ]:
!pip install -q -U "transformers>=4.46,<5" "bitsandbytes>=0.45" accelerate

In [ ]:
import os, glob, json, random, re, time, zipfile, gc
import numpy as np, pandas as pd, torch
ON_KAGGLE = os.path.exists("/kaggle/input")
if ON_KAGGLE:
    DATA_ROOT, OUT_DIR = "/kaggle/input", "/kaggle/working"
else:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = "/content/drive/MyDrive/adaptive-quiz"
    DATA_ROOT, OUT_DIR = "/content/work", f"{DRIVE_DIR}/pilot"
    for z in glob.glob(f"{DRIVE_DIR}/*dataset*.zip"):
        out = os.path.join(DATA_ROOT, os.path.basename(z)[:-4])
        if not os.path.exists(out):
            zipfile.ZipFile(z).extractall(out)
os.makedirs(OUT_DIR, exist_ok=True)
assert torch.cuda.is_available(), "no GPU"
print(torch.cuda.get_device_name(0), "| out:", OUT_DIR)

def find(split):
    hits = sorted(glob.glob(f"{DATA_ROOT}/**/{split}.jsonl", recursive=True), key=lambda p: ("clean" not in p, p))
    assert hits, f"{split}.jsonl not found"; return hits[0]
raw = {s: [json.loads(l) for l in open(find(s), encoding="utf-8")] for s in ["train", "test"]}

# 30 test inputs: 10 per difficulty, spread over branches
rng = random.Random(0)
tests = []
for d in ["easy", "medium", "hard"]:
    pool = [r for r in raw["test"] if r["difficulty"] == d]; rng.shuffle(pool)
    by_b = {}
    for r in pool: by_b.setdefault(r["branch"], []).append(r)
    i = 0
    while sum(1 for t in tests if t["difficulty"] == d) < 10:
        b = list(by_b)[i % len(by_b)]
        if by_b[b]: tests.append(by_b[b].pop())
        i += 1
print(len(tests), "test inputs:", pd.Series([t["branch"] for t in tests]).value_counts().to_dict())

In [ ]:
import re

LETTERS = "ABCD"


def build_input(branch, subject, topic, difficulty, context):
    return (f"generate mcq | branch: {branch} | subject: {subject} | "
            f"topic: {topic} | difficulty: {difficulty} | context: {context}")


_PATTERN = re.compile(
    r"question:\s*(?P<q>.+?)\s*options:\s*"
    r"\(A\)\s*(?P<A>.+?)\s*\(B\)\s*(?P<B>.+?)\s*\(C\)\s*(?P<C>.+?)\s*\(D\)\s*(?P<D>.+?)\s*"
    r"answer:\s*(?P<ans>[ABCD])\b\s*(?:explanation:\s*(?P<exp>.*))?$",
    re.S)


def parse_mcq(text):
    """Parse model output into a dict, or return None if the format is broken."""
    m = _PATTERN.search(text.strip())
    if not m:
        return None
    options = [m.group(l).strip() for l in LETTERS]
    norm = [re.sub(r"\s+", " ", o.lower()) for o in options]
    if any(not o for o in options) or len(set(norm)) < 4:
        return None
    return {"question": m.group("q").strip(), "options": options,
            "answer": m.group("ans"), "explanation": (m.group("exp") or "").strip()}

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
MODELS = ["Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-3B-Instruct", "Qwen/Qwen2.5-7B-Instruct"]
N_CAND = 4
SYSTEM = ("You are a B.Tech exam setter. From the context, write ONE multiple-choice question of the requested difficulty. "
          "Exactly one option is correct; the three wrong options must be different from each other, plausible and clearly wrong. "
          "Check any calculation before answering. Reply in exactly this format: question: <question> options: (A) <option> "
          "(B) <option> (C) <option> (D) <option> answer: <letter> explanation: <one or two sentences>")
shots = [next(r for r in raw["train"] if r["difficulty"] == "easy" and r["branch"] == "CSE"),
         next(r for r in raw["train"] if r["difficulty"] == "hard" and r["branch"] != "CSE")]

def gen_messages(input_text):
    m = [{"role": "system", "content": SYSTEM}]
    for s in shots:
        m += [{"role": "user", "content": s["input_text"]}, {"role": "assistant", "content": s["target_text"]}]
    return m + [{"role": "user", "content": input_text}]

def check_messages(p):
    q = p["question"] + "\n" + "\n".join(f"({l}) {o}" for l, o in zip("ABCD", p["options"]))
    return [{"role": "user", "content": "Answer this multiple-choice question. Reply with only the letter.\n\n" + q}]

def chat(tok, model, msgs_list, n=1, sample=True, max_new=320):
    texts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in msgs_list]
    enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(0)
    kw = dict(do_sample=True, temperature=0.7, top_p=0.9) if sample else dict(do_sample=False)
    with torch.no_grad():
        ids = model.generate(**enc, max_new_tokens=max_new, num_return_sequences=n, pad_token_id=tok.pad_token_id, **kw)
    return tok.batch_decode(ids[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)

def run_model(name):
    tok = AutoTokenizer.from_pretrained(name, padding_side="left")
    model = AutoModelForCausalLM.from_pretrained(name, device_map={"": 0}, torch_dtype=torch.float16,
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                               bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16))
    model.eval(); rows = []; t0 = time.time()
    for i in range(0, len(tests), 2):
        chunk = tests[i:i+2]
        outs = chat(tok, model, [gen_messages(t["input_text"]) for t in chunk], n=N_CAND)
        for j, t in enumerate(chunk):
            cands = [p for p in map(parse_mcq, outs[j*N_CAND:(j+1)*N_CAND]) if p]
            picked, passed = None, False
            if cands:
                answers = chat(tok, model, [check_messages(p) for p in cands], sample=False, max_new=4)
                for p, a in zip(cands, answers):
                    m = re.search(r"[ABCD]", a)
                    if m and m.group(0) == p["answer"]:
                        picked, passed = p, True; break
                picked = picked or cands[0]
            rows.append({"model": name.split("/")[-1], "branch": t["branch"], "subject": t["subject"], "topic": t["topic"],
                         "difficulty": t["difficulty"], "valid": bool(cands), "self_check_pass": passed,
                         "mcq": None if not picked else
                             f"Q: {picked['question']}\n" + "\n".join(f"  ({l}) {o}" for l, o in zip("ABCD", picked["options"]))
                             + f"\n  answer: {picked['answer']}  |  {picked['explanation']}",
                         "raw_first": outs[j*N_CAND][:400], "context": t["context"],
                         "is_correct (you fill Y/N)": ""})
    secs = (time.time() - t0) / len(tests)
    del model; gc.collect(); torch.cuda.empty_cache()
    return rows, secs

## Run all three (each model is downloaded, tested, then freed). 7B download is about 15 GB.

In [ ]:
all_rows, summary = [], []
for name in MODELS:
    print("=" * 30, name, flush=True)
    rows, secs = run_model(name)
    all_rows += rows
    df = pd.DataFrame(rows)
    summary.append({"model": name.split("/")[-1], "valid_%": round(100 * df.valid.mean(), 1),
                    "self_check_pass_%": round(100 * df.self_check_pass.mean(), 1), "sec_per_mcq": round(secs, 1)})
    print(summary[-1], flush=True)
    pd.DataFrame(all_rows).to_csv(f"{OUT_DIR}/pilot_mcqs.csv", index=False)     # saved after every model
summary = pd.DataFrame(summary); summary.to_csv(f"{OUT_DIR}/pilot_summary.csv", index=False)
print(summary.to_string(index=False))

## Read the MCQs side by side (same input, three models)

In [ ]:
df = pd.DataFrame(all_rows)
for k, t in enumerate(tests[:15]):
    print("#" * 100, f"\n[{t['difficulty']}] {t['branch']} | {t['subject']} | {t['topic']}")
    for name in df.model.unique():
        r = df[df.model == name].iloc[k]
        print(f"--- {name}  (self-check {'PASS' if r.self_check_pass else 'fail'})")
        print(r.mcq if r.mcq else "NO VALID MCQ: " + r.raw_first[:200])